# 06 — Classical bath vs quantum bath vs closed TWA (cluster runs)

Reads the output of `jobs/open_submit.sh`:
- data: `data/1D_open/{case}/tau{TAU}/chunk_*.npz` (from `scripts/kz1d/generate_open.py`)
- results: `results/open/{case}_tau{TAU}_t{t}.json` (from `scripts/kz1d/train_open.py`)

| case | EOM | noise | initial state | $\hat t$ |
|---|---|---|---|---|
| classical | $\ddot\phi+\eta\dot\phi-\phi''+V'=\zeta$ | white, $S=2\eta\theta$ | $\phi=\pi=0$ | $\sqrt{2\eta\tau}$ |
| qlangevin | same | $S=\eta\hbar\omega\coth\frac{\hbar\omega}{2T}\,/\,(1+\omega^2/\Omega_c^2)$ | $\phi=\pi=0$ | $\sqrt{2\eta\tau}$ |
| twa | $\ddot\phi-\phi''+V'=0$ | none | Wigner vacuum | $(2\tau)^{1/3}$ |

Question: how early does the final kink pattern become predictable, and how does that depend on the environment?

In [ ]:
import sys, os, glob, json
sys.path.insert(0, "..")
import numpy as np
import matplotlib.pyplot as plt
import src.kz as kz

# ---- inputs ----
TAU      = 128
DATA_DIR = "../data/1D_open"
RES_DIR  = "../results/open"
FIGDIR   = "../figures/1d_open"
CASES    = {"classical": "C0", "qlangevin": "C2", "twa": "C3"}
T_HAT    = {"classical": np.sqrt(2*TAU), "qlangevin": np.sqrt(2*TAU), "twa": (2*TAU)**(1/3)}
FONTSIZE = 16

kz.DOF, kz.dx = 1024, 0.5
os.makedirs(FIGDIR, exist_ok=True)

files = {c: sorted(glob.glob(f"{DATA_DIR}/{c}/tau{TAU}/chunk_*.npz")) for c in CASES}
res   = {c: sorted((json.load(open(f)) for f in glob.glob(f"{RES_DIR}/{c}_tau{TAU}_t*.json")),
                   key=lambda r: r["t"]) for c in CASES}
for c in CASES:
    print(f"{c:>10s}: {len(files[c])} chunks, {len(res[c])} trained times,  t_hat = {T_HAT[c]:.2f}")

## 1. Sanity checks on the data
Kink count at $t=\tau$. KZ predicts $n \propto 1/\hat\xi$: overdamped $\hat\xi=\sqrt2(\tau/2\eta)^{1/4}$, undamped $\hat\xi=(2\tau)^{1/3}$.

In [ ]:
n_def = {c: np.concatenate([np.load(f)["n_defects"] for f in files[c]]) for c in CASES}
lo = min(n.min() for n in n_def.values()); hi = max(n.max() for n in n_def.values())
bins = np.arange(lo, hi + 2) - 0.5

plt.figure(figsize=(10, 5.5))
for c, col in CASES.items():
    plt.hist(n_def[c], bins=bins, alpha=0.5, color=col, label=f"{c}  (mean {n_def[c].mean():.1f}, N={len(n_def[c])})")
plt.xlabel(r"zero crossings at $t = \tau$", fontsize=FONTSIZE)
plt.ylabel("samples", fontsize=FONTSIZE)
plt.xlim(bins.min(), bins.max())
plt.tick_params(labelsize=FONTSIZE - 2)
plt.legend(fontsize=FONTSIZE - 2)
plt.tight_layout()
plt.savefig(f"{FIGDIR}/kink_count_tau{TAU}.png", dpi=200)
plt.show()

In [ ]:
# ---- inputs ----
SAMPLE = 0          # sample within chunk_000
T_MIN  = -40        # lower time limit of the heatmaps

fig, ax = plt.subplots(1, len(CASES), figsize=(8*len(CASES), 7))
rms = {}
for a, c in zip(ax, CASES):
    d  = np.load(files[c][0])
    ph = d["phi_hist"]
    rms[c] = (d["ts"], np.sqrt((ph.astype(np.float64)**2).mean(axis=(0, 2))))
    kz.heatmap(ph[SAMPLE], d["ts"], ax=a)
    a.set_ylim(T_MIN, TAU)
    a.set_title(c, fontsize=FONTSIZE)
    a.xaxis.label.set_size(FONTSIZE); a.yaxis.label.set_size(FONTSIZE)
    a.tick_params(labelsize=FONTSIZE - 2)
    del ph
plt.tight_layout()
plt.savefig(f"{FIGDIR}/heatmaps_tau{TAU}.png", dpi=200)
plt.show()

In [ ]:
# rms field vs time (averaged over chunk_000): when amplification starts in each case
plt.figure(figsize=(10, 6))
for c, col in CASES.items():
    ts, r = rms[c]
    plt.semilogy(ts, r, color=col, lw=2.5, label=c)

# ---- reference lines (edit freely) ----
plt.axvline(16.0, color="C0", ls=":", lw=1.5)
plt.axvline(6.35, color="C3", ls=":", lw=1.5)
plt.axvline(0.0,  color="grey", ls="-", lw=1.0)

plt.xlim(ts.min(), ts.max())
plt.xlabel(r"$t$", fontsize=FONTSIZE)
plt.ylabel(r"$\sqrt{\langle\phi^2\rangle}$", fontsize=FONTSIZE)
plt.tick_params(labelsize=FONTSIZE - 2)
plt.legend(fontsize=FONTSIZE - 2)
plt.tight_layout()
plt.savefig(f"{FIGDIR}/rms_vs_t_tau{TAU}.png", dpi=200)
plt.show()

## 2. Training diagnostics
If a run used all its epochs, early stopping never triggered and the number is an upper bound on the error.

In [ ]:
for c in CASES:
    print(f"{c:>10s}  epochs used: " + "  ".join(f"t={r['t']:g}:{r['n_epochs']}" for r in res[c]))

In [ ]:
# ---- inputs ----
T_SHOW = [-150, -40, 0, 20, 50]

fig, ax = plt.subplots(1, len(CASES), figsize=(7*len(CASES), 6), sharey=True)
colors = plt.cm.viridis(np.linspace(0, 0.9, len(T_SHOW)))
for a, c in zip(ax, CASES):
    by_t = {r["t"]: r for r in res[c]}
    for t, col in zip(T_SHOW, colors):
        if t not in by_t:
            continue
        r = by_t[t]
        ep = np.arange(1, len(r["loss_hist"]) + 1)
        a.plot(ep, r["loss_hist"],     "--", color=col, lw=1.8)
        a.plot(ep, r["val_loss_hist"], "-",  color=col, lw=1.8, label=rf"$t={t}$")
    a.set_title(c, fontsize=FONTSIZE)
    a.set_xlabel("epoch", fontsize=FONTSIZE)
    a.tick_params(labelsize=FONTSIZE - 2)
    a.legend(fontsize=FONTSIZE - 3)
ax[0].set_ylabel("MSE  (dashed train, solid val)", fontsize=FONTSIZE)
plt.tight_layout()
plt.show()

## 3. Headline: predictability vs input time
Solid: RNN sign error on validation. Dashed: sign of the input at time $t$ used as the prediction. 0.5 = no information.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(18, 6.5))
for c, col in CASES.items():
    if not res[c]:
        continue
    T = np.array([r["t"] for r in res[c]])
    ax[0].plot(T, [r["val_loss"] for r in res[c]], "o-", color=col, ms=9, lw=2, label=c)
    ax[1].plot(T, [r["sign_err"] for r in res[c]], "o-", color=col, ms=9, lw=2, label=c + "  (RNN)")
    ax[1].plot(T, [r["base_err"] for r in res[c]], "--", color=col, lw=1.8, label=c + "  (sign of input)")

# ---- reference lines (edit freely) ----
for a in ax:
    a.axvline(16.0, color="C0", ls=":", lw=1.5)
    a.axvline(6.35, color="C3", ls=":", lw=1.5)
ax[1].axhline(0.5, color="grey", ls="-", lw=1.0)
ax[1].text(18.0, 0.45, r"$\hat t_{\rm Langevin}=16$", color="C0", fontsize=FONTSIZE - 2)
ax[1].text(8.0,  0.40, r"$\hat t_{\rm TWA}=6.3$",     color="C3", fontsize=FONTSIZE - 2)

ax[0].set_ylabel("validation MSE (best epoch)", fontsize=FONTSIZE)
ax[1].set_ylabel("per-site sign error", fontsize=FONTSIZE)
ax[1].set_ylim(0, 0.55)
for a in ax:
    a.set_xlim(-185, 70)
    a.set_xlabel(r"input time $t$", fontsize=FONTSIZE)
    a.tick_params(labelsize=FONTSIZE - 2)
    a.legend(fontsize=FONTSIZE - 4)
fig.suptitle(rf"$\tau_Q = {TAU}$", fontsize=FONTSIZE + 2)
plt.tight_layout()
plt.savefig(f"{FIGDIR}/sign_error_vs_t_tau{TAU}.png", dpi=300)
plt.show()

In [ ]:
# same, in units of each case's own freeze-out time
plt.figure(figsize=(10, 6.5))
for c, col in CASES.items():
    if not res[c]:
        continue
    T = np.array([r["t"] for r in res[c]])/T_HAT[c]
    plt.plot(T, [r["sign_err"] for r in res[c]], "o-", color=col, ms=9, lw=2, label=c)

# ---- reference lines (edit freely) ----
plt.axhline(0.5, color="grey", ls="-", lw=1.0)
plt.axvline(0.0, color="grey", ls=":", lw=1.5)
plt.axvline(1.0, color="grey", ls=":", lw=1.5)

plt.xlim(-12, 12)
plt.ylim(0, 0.55)
plt.xlabel(r"$t/\hat t$", fontsize=FONTSIZE)
plt.ylabel("per-site sign error (RNN)", fontsize=FONTSIZE)
plt.tick_params(labelsize=FONTSIZE - 2)
plt.legend(fontsize=FONTSIZE - 2)
plt.tight_layout()
plt.savefig(f"{FIGDIR}/sign_error_vs_t_over_that_tau{TAU}.png", dpi=300)
plt.show()

In [ ]:
# defect-count accuracy
plt.figure(figsize=(10, 6))
for c, col in CASES.items():
    if not res[c]:
        continue
    T = np.array([r["t"] for r in res[c]])
    plt.plot(T, [r["count_acc"] for r in res[c]], "o-", color=col, ms=9, lw=2, label=c)
plt.xlim(-185, 70)
plt.ylim(0, 1.05)
plt.xlabel(r"input time $t$", fontsize=FONTSIZE)
plt.ylabel("defect-count accuracy", fontsize=FONTSIZE)
plt.tick_params(labelsize=FONTSIZE - 2)
plt.legend(fontsize=FONTSIZE - 2)
plt.tight_layout()
plt.show()

## 4. Earliest readable time
First input time at which the RNN sign error drops below each threshold.

In [ ]:
# ---- inputs ----
THRESH = [0.45, 0.4, 0.3, 0.2, 0.1]

print(f"{'case':>10s}  " + "  ".join(f"err<{th:<5g}" for th in THRESH))
for c in CASES:
    T = np.array([r["t"] for r in res[c]]); E = np.array([r["sign_err"] for r in res[c]])
    row = []
    for th in THRESH:
        hit = T[E < th]
        row.append(f"{hit.min():>4g} ({hit.min()/T_HAT[c]:+.1f}t^)" if len(hit) else "    never   ")
    print(f"{c:>10s}  " + "  ".join(row))